In [3]:
import torch
import torch.nn as nn
from torch.nn import functional as F
from dataclasses import dataclass

In [4]:
# Define GPT architecture here.
# The architecture is based on the GPT-2 model, which is a decoder transformer-based model for natural language processing tasks. 
# The model consists of multiple layers of self-attention and feedforward networks, with layer normalization and residual connections.

@dataclass
class ModelConfig:
    vocab_size: int = 50257
    block_size: int = 256
    num_embed: int = 384
    num_heads: int = 6
    num_layers: int = 6
    dropout: float = 0.0


class AttentionHead(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.head_size = config.num_embed // config.num_heads

        self.key = nn.Linear(config.num_embed, self.head_size, bias=False)
        self.query = nn.Linear(config.num_embed, self.head_size, bias=False)
        self.value = nn.Linear(config.num_embed, self.head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(config.block_size, config.block_size)))
        self.dropout = nn.Dropout(config.dropout)
    
    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * self.head_size ** -0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        out = wei @ self.value(x)
        return out
    
class MultiHeadAttention(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.heads = nn.ModuleList([AttentionHead(config) for _ in range(config.num_heads)])
        self.proj = nn.Linear(config.num_embed, config.num_embed)
        self.dropout = nn.Dropout(config.dropout)
    
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        out = self.dropout(out)
        return out

class FeedForward(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.num_embed, 4 * config.num_embed), # expansion factor of 4 as described in the Attention is All You Need paper
            nn.ReLU(),
            nn.Linear(4 * config.num_embed, config.num_embed), # project back to the original embedding size
            nn.Dropout(config.dropout)
        )
    
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.sa = MultiHeadAttention(config)
        self.ffwd = FeedForward(config)
        self.ln1 = nn.LayerNorm(config.num_embed)
        self.ln2 = nn.LayerNorm(config.num_embed)
    
    def forward(self, x):
        # layers: layer normalization -> self-attention -> residual connection -> layer normalization -> feedforward -> residual connection
        x = x + self.sa(self.ln1(x)) # residual connection after self-attention
        x = x + self.ffwd(self.ln2(x)) # residual connection after feedforward
        return x

class GPTModel(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()

        self.config = config

        # We intentionally omit softmax in the final layer because we will use cross-entropy loss which applies softmax internally.
        # This is a common practice in PyTorch to improve efficiency.
        self.token_embedding_table = nn.Embedding(self.config.vocab_size, self.config.num_embed)
        # The fixed sine and cosine positional encodings are replaced with learnable positional embeddings in GPT-2
        self.position_embedding_table = nn.Embedding(self.config.block_size, self.config.num_embed) 
        self.blocks = nn.Sequential(*[Block(self.config) for _ in range(self.config.num_layers)])
        self.ln_f = nn.LayerNorm(self.config.num_embed)
        self.head = nn.Linear(self.config.num_embed, self.config.vocab_size)

        # weight sharing between token embedding and language model head
        self.token_embedding_table.weight = self.head.weight

        #initialize weights
        self.apply(self._init_weights)

    
    def _init_weights(self, module):

        # Initialize weights for linear and embedding layers; not other layrs like LayerNorm etc. 
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02) # mean=0.0, std=0.02 as described in GPT-2 paper
            # Linear layers may have bias, so initialized to zero as described in GPT-2 paper
            if module.bias is not None:
                nn.init.zeros_(module.bias)

        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B,T = idx.shape
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device)) # (T,C); the arange needs to on the same device as idx
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.head(x) # (B,T,vocab_size)

        if targets is None: # target is none in case of inference, we don't need to compute the loss
            loss = None
        else: # target is not none in case of training, we need to compute the loss
            B,T,C = logits.shape # here C is vocab_size
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets) #F.cross_entropy computes the softmax internally, so we don't need to apply softmax to logits before passing it to F.cross_entropy

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is input sequence; shape is (B, T), where B is batch size and T is the length of the input sequence
        for _ in range(max_new_tokens):
            # take only the last block_size tokens. This ensures input not exceeding max block size (context window)
            idx_cond = idx[:, -self.config.block_size:] 
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :] # get only the last time step, shape is (B, vocab_size)
            probs = F.softmax(logits, dim=-1) # apply softmax to get probabilities
            idx_next = torch.multinomial(probs, num_samples=1) # sample from the distribution
            idx = torch.cat((idx, idx_next), dim=1) # append sampled index to the running sequence
        return idx

    
    def count_parameters(self):
        total_params = sum(p.numel() for p in self.parameters())
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        frozen_params = total_params - trainable_params
        return {"total": total_params, "trainable": trainable_params, "frozen": frozen_params}

In [5]:
import torch

class CharTokenizer:
    def __init__(self, text: str):
        self.chars = []
        self.vocab_size = 0
        self.stoi = {}
        self.itos = {}
        self._build_vocab(text)

    def _build_vocab(self, text: str):
        ''' Builds vocabulary lookup tables from the input text corpus'''
        self.chars = sorted(list(set(text)))
        self.vocab_size = len(self.chars)
        self.stoi = {ch:i for i,ch in enumerate(self.chars)}
        self.itos = {i:ch for i,ch in enumerate(self.chars)}

    def encode(self, text):
        """Converts a string of text into a list of integer indices based on the vocabulary."""
        return [self.stoi[ch] for ch in text]

    def decode(self, indices):
        """Converts a list of integer indices back into a string of text based on the vocabulary."""
        return ''.join([self.itos[i] for i in indices])
    def num_tokens(self):
        """Returns the number of unique tokens in the vocabulary."""
        return self.vocab_size



class BatchSampler:
    """Samples sequence windows from a 1D token tensor.

    with_replacement=False: shuffled non-overlapping chunks, each used once per epoch.
    with_replacement=True: random start in 0 .. len(data)-block_size-1.
    """

    def __init__(
        self,
        data: torch.Tensor,
        batch_size: int,
        block_size: int,
        device: str = "cpu",
        shuffle: bool = True, # ignored if with_replacement is True
        with_replacement: bool = False,
    ):
        self.data = data
        self.batch_size = batch_size
        self.block_size = block_size
        self.device = device
        self.shuffle = shuffle
        self.with_replacement = with_replacement

    def __len__(self):
        # number of batches in the epoch (without replacement)
        n_chunks = len(self.data) // (self.block_size + 1)
        return n_chunks // self.batch_size  # e.g. 4 chunks, batch_size=2 → 2

    def __iter__(self):
        if not self.with_replacement:  # without replacement only
            self.cursor = 0
            chunk_len = self.block_size + 1
            n_chunks = len(self.data) // chunk_len
            # start indices of the chunks, used for without replacement only
            starts = torch.arange(n_chunks) * chunk_len  # n_chunks=4, block_size=5 → [0, 6, 12, 18]
            if self.shuffle:
                # shuffle the start indices
                starts = starts[torch.randperm(n_chunks)]  # e.g. [18, 0, 12, 6]
            
            self.starts = starts
        return self

    def __next__(self):
        if self.with_replacement:  # with replacement only
            # index of the start of the next batch
            # batch_size=4 → [3, 0, 8, 1]
            ix = torch.randint(len(self.data) - self.block_size, (self.batch_size,)) 
            return self._windows(ix)

        # without replacement only
        end = self.cursor + self.batch_size
        if end > len(self.starts):
            raise StopIteration
        batch_starts = self.starts[self.cursor:end]  # batch_size=2 → first [0, 6], then [12, 18]
        self.cursor = end
        return self._windows(batch_starts)

    def _windows(self, starts):
        t = torch.arange(self.block_size)  # block_size=5 → [0, 1, 2, 3, 4]
        idx = starts.unsqueeze(1) + t  # batch_size=2, block_size=5: [0, 6] → [[0...4], [6...10]]
        x = self.data[idx]
        y = self.data[idx + 1]
        return x.to(self.device), y.to(self.device)


if __name__ == "__main__":
    # 1. Sample raw text
    sample_text = "First Citizen: Before we proceed any further, hear me speak. All: Speak, speak." * 100

    # 2. Tokenize text into a 1D long tensor
    chars = sorted(list(set(sample_text)))
    stoi = {ch: i for i, ch in enumerate(chars)}
    data = torch.tensor([stoi[c] for c in sample_text], dtype=torch.long)

    # 3. Split 1D tensor upfront (90% train, 10% val)
    n = int(0.9 * len(data))
    train_data, val_data = data[:n], data[n:]

    # 4. Instantiate independent BatchSampler instances
    train_sampler = BatchSampler(train_data, batch_size=16, block_size=32, device="cpu", shuffle=True)
    val_sampler   = BatchSampler(val_data,   batch_size=16, block_size=32, device="cpu", shuffle=False)

    # 5. Get persistent training iterator
    train_iter = iter(train_sampler)

    # 6. Fetch a training batch
    try:
        x_train, y_train = next(train_iter)
    except StopIteration:
        train_iter = iter(train_sampler)
        x_train, y_train = next(train_iter)

    print(f"Train Inputs (X) shape: {x_train.shape} | Targets (Y) shape: {y_train.shape}")

    # 7. Fetch a validation batch
    x_val, y_val = next(iter(val_sampler))
    print(f"Val Inputs   (X) shape: {x_val.shape} | Targets (Y) shape: {y_val.shape}")

Train Inputs (X) shape: torch.Size([16, 32]) | Targets (Y) shape: torch.Size([16, 32])
Val Inputs   (X) shape: torch.Size([16, 32]) | Targets (Y) shape: torch.Size([16, 32])


In [6]:
import torch

@torch.no_grad()
def estimate_loss(model, train_sampler: BatchSampler, val_sampler: BatchSampler, eval_iters: int = 200):
    """Computes average evaluation loss over train and val samplers without mutating state."""
    out = {}
    model.eval()

    samplers = {"train": train_sampler, "val": val_sampler}

    for split_name, sampler in samplers.items():
        losses = torch.zeros(eval_iters)

        actual_steps = 0
        for i, (x, y) in enumerate(sampler):
            if i >= eval_iters:
                break
            logits, loss = model(x, y)
            losses[i] = loss.item()
            actual_steps = i + 1

        out[split_name] = losses[:actual_steps].mean().item()

    model.train()
    return out

In [7]:
import torch

with open("nepali_data/merged_nepali.txt", "r", encoding="utf-8") as f:
    text = f.read()

torch.manual_seed(1337)

tokenizer = CharTokenizer(text)
vocab_size = tokenizer.num_tokens()

print(f"Vocabulary size: {vocab_size}")

data = torch.tensor(tokenizer.encode(text), dtype=torch.long)
n = int(0.9 * len(data))  # 90% for training, 10% for validation
train_data = data[:n]
val_data = data[n:]

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

learn_rate = 3e-4
# device = "cpu"
shuffle_data = True # shuffle data
with_replacement = True # sample batches with replacement

config = ModelConfig(
    vocab_size = vocab_size,
    block_size = 32,
    num_embed= 64,
    num_heads = 4,
    num_layers = 4,
    dropout = 0.0
)


max_iters = 5000
eval_interval = 100
eval_iters = 200


model = GPTModel(config).to(device)

print(model.count_parameters())
print(f"Device: {device}")

optimizer = torch.optim.AdamW(model.parameters(), lr=learn_rate)

train_sampler = BatchSampler(train_data, batch_size=64, block_size=config.block_size, device=device, shuffle=shuffle_data, with_replacement=with_replacement)
eval_train_sampler = BatchSampler(train_data, batch_size=64, block_size=config.block_size, device=device, shuffle=shuffle_data, with_replacement=with_replacement)
val_sampler = BatchSampler(val_data, batch_size=64, block_size=config.block_size, device=device, shuffle=shuffle_data, with_replacement=with_replacement)


train_iter = iter(train_sampler)
for global_step in range(max_iters):

    if global_step % eval_interval == 0 or global_step == max_iters - 1:
        losses = estimate_loss(model, eval_train_sampler, val_sampler, eval_iters=eval_iters)
        print(f"Step {global_step}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    item = next(train_iter, None)
    if item is None:
        train_iter = iter(train_sampler)
        x, y = next(train_iter)
    else:
        x, y = item

    logits, loss = model(x, y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

prompt = "एक्‌ दिन्‌ नारद सत्यलोक पुभिगया"
context = torch.tensor([tokenizer.encode(prompt)], dtype=torch.long, device=device)
print(tokenizer.decode(model.generate(context, max_new_tokens=2000)[0].tolist()))

Vocabulary size: 136
{'total': 210184, 'trainable': 210184, 'frozen': 0}
Device: mps
Step 0: train loss 4.9285, val loss 4.9281
Step 100: train loss 3.3728, val loss 3.4075
Step 200: train loss 3.0101, val loss 3.0588
Step 300: train loss 2.8799, val loss 2.9456
Step 400: train loss 2.8014, val loss 2.8705
Step 500: train loss 2.7336, val loss 2.8152
Step 600: train loss 2.6788, val loss 2.7721
Step 700: train loss 2.6284, val loss 2.7021
Step 800: train loss 2.5783, val loss 2.6513
Step 900: train loss 2.5287, val loss 2.6052
Step 1000: train loss 2.4814, val loss 2.5589
Step 1100: train loss 2.4393, val loss 2.5215
Step 1200: train loss 2.3999, val loss 2.4896
Step 1300: train loss 2.3660, val loss 2.4362
Step 1400: train loss 2.3340, val loss 2.4185
Step 1500: train loss 2.3070, val loss 2.3739
Step 1600: train loss 2.2888, val loss 2.3662
Step 1700: train loss 2.2678, val loss 2.3441
Step 1800: train loss 2.2475, val loss 2.3158
Step 1900: train loss 2.2347, val loss 2.2984
Step 20

In [9]:
prompt = "एक्‌ दिन्‌ नारद सत्यलोक पुभिगया"
context = torch.tensor([tokenizer.encode(prompt)], dtype=torch.long, device=device)
print(tokenizer.decode(model.generate(context, max_new_tokens=1500)[0].tolist()))

एक्‌ दिन्‌ नारद सत्यलोक पुभिगया । १९० १ इसमालाई सत्रीव प्यटा पिएर भरिदीयो । 
क अनेक्‌ शिर पौ कहीं । २७ = 


नेपाली-ह -कष्ट ? इस प्रकार थिन ने बाण भरत | ८६ भा‌ उसन्ह्योगो दै ध्यान हुजस्‌ र लढामेरंका 
उत्तरसूर्यको यदि त हुन्‌ 
यस्ता प्रदाहौजन्‌ पनि तटाइ छन्‌  माथिन लाग्यो ।१२३ 
ता



दरद वीर्‌ दयेष्ट पयँ । 


पहुंगा) जो दुः सिद्‌ भनी, 
यो कह्याडा फङडयाथमै बोली उकाए, 
सरमभिरिलाइ आँडी र उसल्को सुपेको छ । 
सुम्निमालाइ हकमणं आयी सत्धन यही 
आफैर मै जाकर धारण जल्जिकामा कूनै 
आदै परापुरी थाने नहीं । स्वीबाट पर्वन-विचार करका 
सं मेरा) 
पापसे प्रकारसाभी हौ सो शक्तु गर्नुपरिन्थ्यौ कजवार्‌ 
सो जल्दि सवं आज जातः आज वा ब्रह्मणजिवती का बीच देने पासच्यादि सोने जारने! कतन्‌ अधि कि सुँठुस्भरिलाई 'सब सूग्रीवप्‌ ॥। १४॥ 
राम्रती आज याचनी सव्‌ ॥ 
सग्रीता मनि वधूमत्‌ विनाऊ ' 
बाट तुरक्ष्टि पनि। 
गन्छन्‌ दीव्या हूौ उभिया। जब ॥ 




मस्ये कूल कुशना बागत्यो,- ““हिर्किनन अभी तागल मताइदि अ।. 


हेरेको छोडे ! भहांस्रा जसै। 
मंं अहो। 
उदान आई मूखमा छिप्यो । सुम्निमाका मध्न थालकर जाउँहि सव 
टक्कारन्‌ पौौ । ३३ वात चुक